In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os
from delta.tables import DeltaTable

from datetime import datetime
import time
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from pyspark.sql.window import Window
from datetime import datetime, timedelta


In [3]:
from pyspark.sql.types import StructType, StructField, StringType

In [4]:
import boto3

In [5]:
region = os.getenv('AWS_REGION')
aws_access_key = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_key = os.getenv('AWS_SECRET_ACCESS_KEY')
NAO_DEFINIDO = os.getenv('NAO_DEFINIDO', '99999999')

In [6]:
s3_client = boto3.client(
            's3',
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
            region_name='sa-east-1'
        )

In [7]:


user_temp = os.path.expanduser("~/tmp")
os.makedirs(f"{user_temp}/spark", exist_ok=True)
os.makedirs(f"{user_temp}/hadoop", exist_ok=True)
os.makedirs(f"{user_temp}/s3a", exist_ok=True)

# Configurar variáveis de ambiente para Python
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession
    .builder
    .master("local[6]")
    .appName("Silver_Processing")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")    
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB")
    .config("spark.sql.files.maxPartitionBytes", "256MB")
    # Suprimir warnings
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.500")
    # Configurações para Snappy
    .config("spark.executor.extraJavaOptions", "-Djava.library.path=/usr/lib")
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=/usr/lib")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.parquet.writeLegacyFormat", "true")
    .config("spark.sql.legacy.parquet.nanosAsLong", "true")
    # Configurações de compressão - usar GZIP em vez de Snappy
    .config("spark.sql.parquet.compression.codec", "gzip")
    .config("spark.io.compression.codec", "lz4")
    # Python worker configs
    
    .config("spark.python.worker.reuse", "true")
    .config("spark.python.worker.memory", "1g")  # Reduzido de 2g para 1g
    .config("spark.executor.cores", "6")         # Ajustado para usar 6 cores
    .config("spark.default.parallelism", "12")   # 2x número de cores
    .config("spark.sql.shuffle.partitions", "100") # Reduzido de 200 para 100
    # Configurações de memória mais conservadoras
    .config("spark.executor.memoryFraction", "0.8")
    .config("spark.storage.memoryFraction", "0.5")           
    # Suprimir logs de UI
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")    
    # Diretórios
    .config("spark.local.dir", f"{user_temp}/spark")
    # S3 configs
    .config("spark.hadoop.fs.s3a.access.key", aws_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", aws_secret_key)
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "array")
    # Configurações adicionais de compressão
    .config("spark.hadoop.mapreduce.output.fileoutputformat.compress", "true")
    .config("spark.hadoop.mapreduce.output.fileoutputformat.compress.codec", "org.apache.hadoop.io.compress.GzipCodec")
    .getOrCreate()
)


# Configurar níveis de log mais restritivos
spark.sparkContext.setLogLevel("ERROR")

# Desabilitar logs específicos do Spark
import logging
logging.getLogger("org").setLevel(logging.ERROR)
logging.getLogger("akka").setLevel(logging.ERROR)
logging.getLogger("py4j").setLevel(logging.ERROR)

# Configurar log4j para suprimir todos os logs desnecessários
try:
    loggers_to_suppress = [
        "org.apache.spark.SparkConf",
        "org.apache.spark.util.Utils", 
        "org.apache.hadoop.metrics2.impl.MetricsConfig",
        "org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec",
        "org.apache.spark.sql.catalyst.optimizer.PlanChangeLogger",
        "org.apache.spark.util.ShutdownHookManager"
    ]

    for logger_name in loggers_to_suppress:
        spark.sparkContext._jvm.org.apache.log4j.Logger.getLogger(logger_name).setLevel(
            spark.sparkContext._jvm.org.apache.log4j.Level.OFF
        )
except Exception as e:
    print(f"Aviso: Não foi possível configurar alguns loggers: {e}")

In [8]:
silver_path = os.getenv('SILVER_PATH')
security_code = os.getenv('SECURITY_CODE')
gold_path = os.getenv('GOLD_PATH')
NAO_DEFINIDO = os.getenv('NAO_DEFINIDO', '99999999')

In [9]:
delta_path = 's3a://brid-silver/5037/DIM_FILIAL/'

In [10]:
delta_table = DeltaTable.forPath(spark, delta_path)

In [11]:
properties = delta_table.detail().select("properties").collect()[0][0]

In [12]:
cdf_enabled = properties.get("delta.enableChangeDataFeed", "false")

In [13]:
cdf_enabled

'true'

In [14]:
df_changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .load(delta_path)

In [16]:
df_changes.printSchema()

root
 |-- COD_FILIAL: string (nullable = true)
 |-- DESC_FILIAL: string (nullable = true)
 |-- DESC_REGIONAL: string (nullable = true)
 |-- DESC_MICRORREGIAO: string (nullable = true)
 |-- DESC_UF: string (nullable = true)
 |-- TEL_AA: string (nullable = true)
 |-- FAX_AA: string (nullable = true)
 |-- CNPJ: string (nullable = true)
 |-- IE: string (nullable = true)
 |-- NIRE: string (nullable = true)
 |-- END_COMPLETO: string (nullable = true)
 |-- DT_PRI_NF_PROTHEUS: string (nullable = true)
 |-- extract_dt: timestamp (nullable = true)
 |-- hash_value: string (nullable = true)
 |-- data_source: string (nullable = true)
 |-- ativo: byte (nullable = true)
 |-- data_inicio: timestamp (nullable = true)
 |-- data_fim: timestamp (nullable = true)
 |-- _change_type: string (nullable = true)
 |-- _commit_version: long (nullable = true)
 |-- _commit_timestamp: timestamp (nullable = true)



In [16]:
df_changes.show()

+----------+---------+--------+--------------+-----------------+---------------+--------------------+--------------------+------------+-------------+--------+--------------------+--------+-------------+---------------+------------------+--------------------+--------------+------------------+---------------+----------+----------------+-----------+-------------+------------+------------+--------------------+------------+---------------+------------------+--------------------+-----------+--------------+---------------+---------------+--------------------+------------------+------------+--------------------+-------------+--------------------+-------------------+--------------------+--------------------+-----+--------------------+--------+------------+---------------+-------------------+
|COD_FILIAL|    NR_NF|SERIE_NF|COD_FABRICANTE|ESPECIE_DOCUMENTO|CPNJ_EMISSOR_NF|           CHAVE_NFE|             TIPO_NF|DATA_EMISSAO|    NF_ORIGEM|COD_CFOP|           DESC_CFOP|COD_IBGE|TIPO_OPERACAO|    

In [11]:
def _table_exists( spark,delta_path):
    try:
        DeltaTable.forPath(spark, delta_path)
        print("Tabela Delta existe - processamento incremental")
        return True
    except:
        print("Tabela Delta não existe - primeira carga")
        return False

In [8]:

def processa_gravacao(delta_path, df_atual, tabela):   
    
    table_exists = _table_exists(spark, delta_path)
    
    if not table_exists:
        print('não existe, primeira carga')
        df_atual.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").save(delta_path)
        spark.sql(f"OPTIMIZE delta.`{delta_path}`")
    else:
        print('existe, processamento incremental')       

        s3_path = f"{gold_path}/{tabela}"  
        print(f'carregando tabela delta {s3_path}')
        df_delta_table = spark.read.format("delta").load(s3_path)
        df_delta_comparacao = df_delta_table.filter(df_delta_table.ativo == True)
        df_delta_comparacao = df_delta_comparacao.drop("data_inicio")  
        df_delta_comparacao = df_delta_comparacao.drop("data_fim")
        df_delta_comparacao = df_delta_comparacao.drop("ativo")
        #df_delta_comparacao = df_delta_comparacao.drop("hash_value")
        
        df_atual_comparacao = df_atual.drop("data_inicio")
        df_atual_comparacao = df_atual_comparacao.drop("data_fim")
        df_atual_comparacao = df_atual_comparacao.drop("ativo")


        df_insert = df_atual_comparacao.subtract(df_delta_comparacao)
        df_delete = df_delta_comparacao.subtract(df_atual_comparacao)
        total_linhas_df = df_atual_comparacao.count()
        total_linhas_atual = df_delta_comparacao.count()
        total_linhas = df_insert.count()
        total_linhas_delete = df_delete.count()

        print(f'Total de linhas delta table: {total_linhas_atual} - Total de Linhas carga atual: {total_linhas_df} - Total de Linhas novas: {total_linhas} - Total de linhas deletadas: {total_linhas_delete}')

        if total_linhas_delete > 0:    
            
            
            condicao = f"atual.hash_value = delecao.hash_value"
            print(f'condição {condicao}')

            print('carregando tabela delta para update')
            s3_path = f"{gold_path}/{tabela}"                

            delta_table = DeltaTable.forPath(spark, s3_path)
            print('tabela delta carregada')

            delta_table.alias("atual").merge(
                df_delete.alias("delecao"),
                condicao
            ).whenMatchedUpdate(
                    set = { "ativo": "'False'",
                            "data_fim":"current_timestamp()"
                                }  # Definir a atualização
            ).execute()

        if total_linhas > 0:
            print('iniciando atualização tabela delta')       

            s3_path = f"{gold_path}/{tabela}"      

            df_status = df_atual.select("hash_value", "data_inicio", "data_fim", "ativo")
            df_status = df_status.filter(df_status.ativo=='True')    
            df_insert = df_insert.join(df_status, on=["hash_value"], how='inner')
            

            df_insert.write.format("delta").mode("append").save(s3_path)

            print('inserção finalizada')

In [25]:
def exportar_parquet(gold_table_name, folder, nome_arquivo):
            gold_path = 's3a://brid-gold/026'
            gold_path = f"{gold_path}/{gold_table_name}"
            
            if not _table_exists(spark, gold_path):
                print(f"Tabela Gold {gold_table_name} não encontrada.")
                return False
            
            # Ler e limpar dados
            df = spark.read.format("delta").load(gold_path)
            
            # Filtrar apenas registros ativos se a coluna existir
            if 'ativo' in df.columns:
                df = df.filter(col('ativo') == True)
                df = df.drop("data_inicio", "data_fim", "ativo", "hash_value")
            else:
                # Para tabelas sem controle de versionamento (como CALENDARIO)
                columns_to_drop = [c for c in ["data_inicio", "data_fim", "ativo", "hash_value"] if c in df.columns]
                if columns_to_drop:
                    df = df.drop(*columns_to_drop)
            
            # Nome do arquivo
            if not nome_arquivo:                
                nome_arquivo = f"{gold_table_name.lower()}"
            
            bucket = 'brid-gold/026'
            # Exportar usando boto3
            temp_path = f"s3a://{bucket}/temp/{nome_arquivo}"
            df.coalesce(1).write.mode("overwrite").parquet(temp_path)
            
            # Renomear arquivo final
            bucket_name = bucket.split('/')[0]
            cliente_prefix = bucket.split('/')[1] if '/' in bucket else ''
            temp_prefix = f"{cliente_prefix}/temp/{nome_arquivo}/"
            
            response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=temp_prefix)
            
            for obj in response.get('Contents', []):
                if obj['Key'].endswith('.parquet'):
                    copy_source = {'Bucket': bucket_name, 'Key': obj['Key']}
                    final_key = f"{cliente_prefix}/parquet/{folder}/{nome_arquivo}.parquet"
                    s3_client.copy_object(CopySource=copy_source, Bucket=bucket_name, Key=final_key)
                    s3_client.delete_object(Bucket=bucket_name, Key=obj['Key'])
                    break
            

In [29]:
exportar_parquet('FAMILIA_SUPRIMENTOS', 'Suprimentos', 'Familia')

Tabela Delta existe - processamento incremental


In [9]:
df_novo = spark.read.format("delta").load(f"{gold_path}/FAMILIA_SUPRIMENTOS")

In [11]:
df_novo = df_novo.distinct()

In [18]:
df_novo.count()

73723

In [10]:
df_novo.show()

+----------------+-------------+--------------------+--------------------+--------+-----+------------+
|FamiliaDescricao|FamiliaCodigo|          hash_value|         data_inicio|data_fim|ativo|SecurityCode|
+----------------+-------------+--------------------+--------------------+--------+-----+------------+
|       74I77IPRO|            1|fd044a5a4b7e8625d...|2025-12-18 10:26:...|    NULL| true|         026|
|   74I77RSF IPRO|            2|b4876483aa49b6140...|2025-12-18 10:26:...|    NULL| true|         026|
|   74I78RSF IPRO|            3|7bc800218844f5a5b...|2025-12-18 10:26:...|    NULL| true|         026|
|   75I77RSF IPRO|            4|977c75e175a4f7c57...|2025-12-18 10:26:...|    NULL| true|         026|
|         797IPRO|            5|c6074396a1b07790b...|2025-12-18 10:26:...|    NULL| true|         026|
|        8473 RSF|            6|3571cf0bd41250c08...|2025-12-18 10:26:...|    NULL| true|         026|
|    8579RSF IPRO|            7|0abef05a056b6ae06...|2025-12-18 10:26:...

In [18]:
df_novo = df_novo.drop("ProdutoFamilia", "FamiliaCodigo")

In [30]:
df_novo = df_novo.drop("hash_value", "data_inicio", "data_fim", "ativo", "SecurityCode", "DataProcessamento")

In [27]:
df_parquet = spark.read.format("parquet").load(f"{gold_path}/parquet/CONTAS_RECEBER.parquet")


In [29]:
df_parquet.count()

12326

In [28]:
df_parquet.show()

+------------------+-------------------+------------------------+-------------------------+--------------------------+--------------------------+--------------------+--------------------+-------------------------------+-------------------------------+--------------------------------+------------------------+-------------------------+-----------------+--------------------+---------------------------------+-------------------------+-----------------------------+--------------------+------------------------------+---------------------+---------------------------+----------------------------+--------------------------+-----------------------+-------------+-------------------------+--------------------------+----------------------+--------------------+--------------------------------+---------------------------+--------------------+-----------------------+--------------------------+-------------------+-------------------------------+---------------------+------------------------+-----------

In [20]:
df_parquet = df_parquet.drop("ProdutoFamilia", "FamiliaCodigo")

In [56]:
df_parquet = df_parquet.distinct()

In [31]:
df_insert = df_parquet.subtract(df_novo)
df_delete = df_novo.subtract(df_parquet)

In [58]:
df_delete.show()

+----------------------+-------------------------+
|ClientePrincipalCodigo|ClientePrincipalDescricao|
+----------------------+-------------------------+
+----------------------+-------------------------+



In [32]:
df_insert.show()

+------------------+-------------------+------------------------+-------------------------+--------------------------+--------------------------+--------------------+--------------------+-------------------------------+-------------------------------+--------------------------------+------------------------+-------------------------+-----------------+--------------------+---------------------------------+-------------------------+-----------------------------+--------------------+------------------------------+---------------------+---------------------------+----------------------------+--------------------------+-----------------------+-------------+-------------------------+--------------------------+----------------------+--------------------+--------------------------------+---------------------------+--------------------+-----------------------+--------------------------+-------------------+-------------------------------+---------------------+------------------------+-----------

In [69]:
print(f'insert = {df_insert.count()} - delete = {df_delete.count()}')

insert = 0 - delete = 0


In [152]:
df_delete.filter(df_delete.VendedorCodigo=='S-139').show()

+--------------+--------------+----------------------------+-----------------------------+--------------------+-------------+--------------------+----------------------+----------------------+-------------------+----------------+---------------+--------------------+-----------+----------+-------------+--------------------------+--------------------+------------------+-------------------+-------------------------+--------------------------+---------------+----------------+-------------+
|VendedorCodigo|VendedorFilial|VendedorEnderecoMunicipioERP|VendedorEnderecoMunicipioIBGE|   VendedorDescricao|VendedorAtivo|    VendedorEndereco|VendedorEnderecoNumero|VendedorEnderecoBairro|VendedorEnderecoCEP|VendedorTelefone|VendedorCelular|       VendedorEmail|VendedorCPF|VendedorRG|VendedorConta|VendedorContaIdentificador|VendedorContaAgencia|VendedorContaBanco|VendedorContaNumero|VendedorContaMunicipioERP|VendedorContaMunicipioIBGE|VendedorGerente|VendedorSuperior|VendedorCargo|
+--------------+--

In [154]:
df_insert.filter(df_insert.VendedorCodigo=='S-139').show()

+--------------+--------------+----------------------------+-----------------------------+--------------------+-------------+--------------------+----------------------+----------------------+-------------------+----------------+---------------+--------------------+-----------+----------+-------------+--------------------------+--------------------+------------------+-------------------+-------------------------+--------------------------+---------------+----------------+-------------+
|VendedorCodigo|VendedorFilial|VendedorEnderecoMunicipioERP|VendedorEnderecoMunicipioIBGE|   VendedorDescricao|VendedorAtivo|    VendedorEndereco|VendedorEnderecoNumero|VendedorEnderecoBairro|VendedorEnderecoCEP|VendedorTelefone|VendedorCelular|       VendedorEmail|VendedorCPF|VendedorRG|VendedorConta|VendedorContaIdentificador|VendedorContaAgencia|VendedorContaBanco|VendedorContaNumero|VendedorContaMunicipioERP|VendedorContaMunicipioIBGE|VendedorGerente|VendedorSuperior|VendedorCargo|
+--------------+--

In [109]:
produto_df_parquet.show()

+-------------+--------------------+---------------------+-----------------------+--------------+---------------+------------+-----------+-----------------+---------------------+
|ProdutoCodigo|    ProdutoDescricao|ProdutoCodigoSyngenta|ProdutoFamiliaDescricao|ProdutoFamilia|ProdutoSubgrupo|ProdutoGrupo|ProdutoTipo|ProdutoFabricante|ProdutoFatorConversao|
+-------------+--------------------+---------------------+-----------------------+--------------+---------------+------------+-----------+-----------------+---------------------+
|    S-0002256|SEM. SOJA SYN1367...|                 NULL|          SYN13671 IPRO|         S-268|      S-2-377-4|         S-2|        S-4|            S-209|          1.000000000|
|    S-0002297|BRAVONIL 500 4X5L...|              1005209|               SYNGENTA|         S-273|        S-1-3-3|         S-1|        S-3|            S-146|          5.000000000|
|    S-0002828|  AVICTA 500 FS 4X5L|   000000000000033481|               SYNGENTA|         S-302|        